<a href="https://colab.research.google.com/github/Yennybel01/ALGORITMOS-GENETICOS-EN-MACHINE-LEARNING/blob/main/FEATURE_SELECTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
==============================================================================
ALGORITMO GENÉTICO PARA SELECCIÓN DE CARACTERÍSTICAS (Feature Selection)
Dataset: Mushroom Classification (UCI / Kaggle)
==============================================================================
"""

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import random

# ─────────────────────────────────────────────────────────────────────────────
# 0. CARGA DEL DATASET REAL
# ─────────────────────────────────────────────────────────────────────────────
df = pd.read_csv('mushrooms.csv')
le = LabelEncoder()
for col in df.columns:
    df[col] = le.fit_transform(df[col])

X = df.drop('class', axis=1).values
y = df['class'].values

FEATURE_NAMES = list(df.drop('class', axis=1).columns)
n_features    = X.shape[1]
cv            = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ─────────────────────────────────────────────────────────────────────────────
# 1. REPRESENTACIÓN DE LA POBLACIÓN (cromosomas binarios)
# ─────────────────────────────────────────────────────────────────────────────
# Cada cromosoma = vector binario de longitud n_features (22 genes)
# Gen = 1 → característica SELECCIONADA
# Gen = 0 → característica DESCARTADA
#
# Ejemplo visual:
#   Posición: [ 0         1           2          3        4     ... 21      ]
#   Feature:  [ cap-shape cap-surface cap-color  bruises  odor  ... habitat ]
#   Cromosoma:[ 1         0           1          1        0     ... 1       ]
#               ↑ usa                 ↑ usa      ↑ usa          ↑ usa

np.random.seed(0)
ejemplo_cromosoma = np.random.randint(0, 2, n_features)
print("=" * 60)
print(" REPRESENTACIÓN DEL CROMOSOMA")
print("=" * 60)
print(f"  Genes (22 bits): {ejemplo_cromosoma.tolist()}")
print(f"  Features activas ({ejemplo_cromosoma.sum()}): "
      f"{[FEATURE_NAMES[i] for i, g in enumerate(ejemplo_cromosoma) if g == 1]}")
print(f"  Features descartadas ({n_features - ejemplo_cromosoma.sum()}): "
      f"{[FEATURE_NAMES[i] for i, g in enumerate(ejemplo_cromosoma) if g == 0]}")
print()

# ─────────────────────────────────────────────────────────────────────────────
# 2. INICIALIZACIÓN
# ─────────────────────────────────────────────────────────────────────────────
def initialize_population(pop_size, n_genes, min_features=2):
    """
    Genera pop_size cromosomas binarios aleatorios.
    Garantiza al menos min_features genes activos por cromosoma.
    """
    population = []
    for _ in range(pop_size):
        while True:
            chrom = np.random.randint(0, 2, n_genes)
            if chrom.sum() >= min_features:
                break
        population.append(chrom)
    return population

# ─────────────────────────────────────────────────────────────────────────────
# 3. FUNCIÓN DE APTITUD (fitness)
# ─────────────────────────────────────────────────────────────────────────────
def fitness(chromosome, X, y, cv, alpha=0.01):
    """
    fitness = accuracy_CV - alpha * (proporción de features usadas)
    Penaliza usar demasiadas características → prefiere soluciones compactas.
    """
    selected = np.where(chromosome == 1)[0]
    if len(selected) == 0:
        return 0.0
    X_sel  = X[:, selected]
    model  = RandomForestClassifier(n_estimators=30, random_state=42)
    scores = cross_val_score(model, X_sel, y, cv=cv, scoring="accuracy")
    acc     = scores.mean()
    penalty = alpha * (len(selected) / len(chromosome))
    return acc - penalty

# ─────────────────────────────────────────────────────────────────────────────
# 4. SELECCIÓN (torneo binario)
# ─────────────────────────────────────────────────────────────────────────────
def tournament_selection(population, fitnesses, k=3):
    """
    Selecciona un individuo eligiendo k candidatos al azar
    y devolviendo el de mayor fitness (torneo).
    """
    idx  = random.sample(range(len(population)), k)
    best = max(idx, key=lambda i: fitnesses[i])
    return population[best].copy()

# ─────────────────────────────────────────────────────────────────────────────
# 5. CRUZAMIENTO (un punto)
# ─────────────────────────────────────────────────────────────────────────────
def single_point_crossover(parent1, parent2, prob=0.8):
    """
    Cruzamiento de un punto: elige un punto aleatorio y
    combina los segmentos de ambos padres para crear dos hijos.
    """
    if random.random() < prob:
        point  = random.randint(1, len(parent1) - 1)
        child1 = np.concatenate([parent1[:point], parent2[point:]])
        child2 = np.concatenate([parent2[:point], parent1[point:]])
        return child1, child2
    return parent1.copy(), parent2.copy()

# ─────────────────────────────────────────────────────────────────────────────
# 6. MUTACIÓN (bit-flip)
# ─────────────────────────────────────────────────────────────────────────────
def bit_flip_mutation(chromosome, prob=0.05):
    """
    Invierte cada bit del cromosoma con probabilidad prob.
    Garantiza al menos 1 feature activa tras la mutación.
    """
    mutant = chromosome.copy()
    for i in range(len(mutant)):
        if random.random() < prob:
            mutant[i] = 1 - mutant[i]   # 0 → 1  ó  1 → 0
    if mutant.sum() == 0:
        mutant[random.randint(0, len(mutant) - 1)] = 1
    return mutant

# ─────────────────────────────────────────────────────────────────────────────
# 7. CICLO PRINCIPAL DEL ALGORITMO GENÉTICO
# ─────────────────────────────────────────────────────────────────────────────
def genetic_algorithm_feature_selection(
    X, y, pop_size=20, n_generations=15,
    crossover_prob=0.8, mutation_prob=0.05, elitism=2, verbose=True
):
    n_genes    = X.shape[1]
    population = initialize_population(pop_size, n_genes)

    best_fitness_history = []
    avg_fitness_history  = []
    best_chromosome      = None
    best_fit             = -np.inf

    print(f"{'Gen':>4} | {'Mejor Fit':>10} | {'Prom Fit':>10} | {'# Features':>10}")
    print("-" * 48)

    for gen in range(n_generations):

        # --- Evaluar aptitud de toda la población ---
        fitnesses    = [fitness(c, X, y, cv) for c in population]
        gen_best_fit = max(fitnesses)
        gen_avg_fit  = np.mean(fitnesses)
        best_fitness_history.append(gen_best_fit)
        avg_fitness_history.append(gen_avg_fit)

        # --- Guardar el mejor individuo global ---
        best_idx = np.argmax(fitnesses)
        if fitnesses[best_idx] > best_fit:
            best_fit        = fitnesses[best_idx]
            best_chromosome = population[best_idx].copy()

        n_sel = int(best_chromosome.sum())
        if verbose:
            print(f"{gen+1:>4} | {gen_best_fit:>10.4f} | {gen_avg_fit:>10.4f} | {n_sel:>10}")

        # --- Elitismo: conservar los k mejores ---
        elite_idx = np.argsort(fitnesses)[-elitism:]
        new_pop   = [population[i].copy() for i in elite_idx]

        # --- Generar nueva población ---
        while len(new_pop) < pop_size:
            p1      = tournament_selection(population, fitnesses)   # selección
            p2      = tournament_selection(population, fitnesses)   # selección
            c1, c2  = single_point_crossover(p1, p2, crossover_prob)  # cruzamiento
            c1      = bit_flip_mutation(c1, mutation_prob)          # mutación
            c2      = bit_flip_mutation(c2, mutation_prob)          # mutación
            new_pop.extend([c1, c2])

        population = new_pop[:pop_size]

    # ── TERMINACIÓN ──────────────────────────────────────────────────────────
    selected_features = np.where(best_chromosome == 1)[0]
    print(f"\n✔  Mejor cromosoma encontrado:")
    print(f"   Cromosoma (bits): {best_chromosome.tolist()}")
    print(f"   Características seleccionadas ({len(selected_features)}/{n_genes}): "
          f"{[FEATURE_NAMES[i] for i in selected_features]}")
    print(f"   Fitness final: {best_fit:.4f}")

    return best_chromosome, best_fitness_history, avg_fitness_history

# ─────────────────────────────────────────────────────────────────────────────
# EJECUCIÓN
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 60)
    print(" AG - SELECCIÓN DE CARACTERÍSTICAS | Mushroom Dataset")
    print("=" * 60)

    best_chrom, best_hist, avg_hist = genetic_algorithm_feature_selection(
        X, y, pop_size=20, n_generations=15,
        crossover_prob=0.8, mutation_prob=0.05, elitism=2, verbose=True
    )

    # --- Comparación final: todas las features vs las seleccionadas por el AG ---
    selected   = np.where(best_chrom == 1)[0]
    model_full = RandomForestClassifier(n_estimators=100, random_state=42)
    model_sel  = RandomForestClassifier(n_estimators=100, random_state=42)

    acc_full = cross_val_score(model_full, X,              y, cv=cv).mean()
    acc_sel  = cross_val_score(model_sel,  X[:, selected], y, cv=cv).mean()

    print(f"\n📊 Comparación final:")
    print(f"   Accuracy (todas las {n_features} features): {acc_full:.4f}")
    print(f"   Accuracy ({len(selected)} features AG):     {acc_sel:.4f}")
    print(f"   Reducción de dimensionalidad: "
          f"{(1 - len(selected)/n_features)*100:.1f}%")

 REPRESENTACIÓN DEL CROMOSOMA
  Genes (22 bits): [0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1]
  Features activas (12): ['cap-surface', 'cap-color', 'odor', 'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color', 'stalk-shape', 'stalk-root', 'stalk-color-above-ring', 'spore-print-color', 'habitat']
  Features descartadas (10): ['cap-shape', 'bruises', 'stalk-surface-above-ring', 'stalk-surface-below-ring', 'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number', 'ring-type', 'population']

 AG - SELECCIÓN DE CARACTERÍSTICAS | Mushroom Dataset
 Gen |  Mejor Fit |   Prom Fit | # Features
------------------------------------------------
   1 |     0.9968 |     0.9918 |          7
   2 |     0.9968 |     0.9949 |          7
   3 |     0.9968 |     0.9936 |          7
   4 |     0.9973 |     0.9936 |          6
   5 |     0.9973 |     0.9910 |          6
   6 |     0.9973 |     0.9945 |          6
   7 |     0.9977 |     0.9968 |          5
   8 |     0